# Medical RAG Chatbot Changes

Features covered here:
1. Runnable app pipeline (mirrored into `app.py`)
2. Conversational memory (history-aware retriever)
3. Source citations (page numbers returned with every answer)
4. Central config (Feature 4 lives in `src/config.py`)
5. Hybrid retrieval (BM25 keyword + dense vectors)
6. Better chunking (larger, cleaned chunks)
7. Multi-query expansion
8. Upgraded embedding model (BGE)
9. Medical guardrails (emergency gate + safety prompt)
10. Grounding / anti-hallucination check

## 0. Setup
Run from the project root. (The notebook lives in `research/`, so we step up one level.)

In [1]:
import os
if os.path.basename(os.getcwd()) == "research":
    os.chdir("..")
print("Working dir:", os.getcwd())

Working dir: c:\Users\ASUS\Desktop\Projects\Medical_RAG\Medical-RAG-Chatbot


In [ ]:
# One-time installs (uncomment if needed)
# %pip install -U langchain langchain-community langchain-text-splitters \
#     langchain-pinecone langchain-groq sentence-transformers rank-bm25 \
#     pinecone pypdf python-dotenv flask

### Central configuration

In [2]:
from dotenv import load_dotenv
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY or ""
os.environ["GROQ_API_KEY"] = GROQ_API_KEY or ""

DATA_DIR = "./data"
INDEX_NAME = "medical-rag-chatbot"

EMBED_MODEL = "BAAI/bge-small-en-v1.5"   # Feature 8: upgraded embeddings (384-dim, same as before)
EMBED_DIM = 384
RERANKER_MODEL = "BAAI/bge-reranker-base"
LLM_MODEL = "llama-3.3-70b-versatile"
LLM_TEMPERATURE = 0.3

CHUNK_SIZE = 800        # Feature 6: larger, more coherent chunks
CHUNK_OVERLAP = 120

DENSE_K = 10
SPARSE_K = 10
RERANK_TOP_N = 4

USE_HYBRID = True       # Feature 5
USE_MULTI_QUERY = True  # Feature 7
GROUNDING_CHECK = True  # Feature 10

print("Config loaded. Embedding model:", EMBED_MODEL)

Config loaded. Embedding model: BAAI/bge-small-en-v1.5


### Load, clean and chunk the data

In [3]:
from typing import List
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema import Document

def load_pdfs(data_dir):
    loader = DirectoryLoader(data_dir, glob="*.pdf", loader_cls=PyPDFLoader)
    return loader.load()

def filter_docs(docs: List[Document]) -> List[Document]:
    # Keep source AND page so we can show citations (Feature 3)
    out = []
    for d in docs:
        out.append(Document(
            page_content=d.page_content,
            metadata={"source": d.metadata.get("source"), "page": d.metadata.get("page")},
        ))
    return out

def split_docs(docs: List[Document]) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    chunks = splitter.split_documents(docs)
    # The PDF has blank pages and header-only pages; drop near-empty chunks.
    return [c for c in chunks if c.page_content and len(c.page_content.strip()) > 30]

c:\Users\ASUS\anaconda3\envs\medical-rag\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw = load_pdfs(DATA_DIR)
clean = filter_docs(raw)
chunks = split_docs(clean)
print("Pages:", len(raw), "-> usable chunks:", len(chunks))
chunks[0]

Pages: 637 -> usable chunks: 4122


Document(metadata={'source': 'data\\Medical_book.pdf', 'page': 1}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION')

## Upgraded embedding model (BGE)

In [5]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

def embeddings_init():
    return HuggingFaceBgeEmbeddings(
        model_name=EMBED_MODEL,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
        query_instruction="Represent this sentence for searching relevant passages:",
    )

embeddings = embeddings_init()
print("Embedding dim:", len(embeddings.embed_query("test")))

C:\Users\ASUS\AppData\Local\Temp\ipykernel_5268\1762257261.py:4: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceBgeEmbeddings(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Embedding dim: 384


## Pinecone index

In [7]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=PINECONE_API_KEY)
if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME, dimension=EMBED_DIM, metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
print("Index ready:", INDEX_NAME)

Index ready: medical-rag-chatbot


Only run above cell once.

In [8]:
docsearch = PineconeVectorStore.from_documents(
    documents=chunks, embedding=embeddings, index_name=INDEX_NAME,
)
print("Uploaded", len(chunks), "chunks")

Uploaded 4122 chunks


In [9]:
docsearch = PineconeVectorStore.from_existing_index(index_name=INDEX_NAME, embedding=embeddings)

In [10]:
from langchain_groq import ChatGroq
llm = ChatGroq(model=LLM_MODEL, temperature=LLM_TEMPERATURE)

## Hybrid retrieval + cross-encoder reranking

In [11]:
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

def build_base_retriever(docsearch, chunks):
    dense = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": DENSE_K})
    if not USE_HYBRID or not chunks:
        return dense
    bm25 = BM25Retriever.from_documents(chunks)
    bm25.k = SPARSE_K
    return EnsembleRetriever(retrievers=[bm25, dense], weights=[0.4, 0.6])

def build_reranker():
    model = HuggingFaceCrossEncoder(model_name=RERANKER_MODEL)
    return CrossEncoderReranker(model=model, top_n=RERANK_TOP_N)

## Multi-query expansion

In [12]:
from langchain.retrievers.multi_query import MultiQueryRetriever

def build_retriever(docsearch, chunks, llm):
    base = build_base_retriever(docsearch, chunks)
    if USE_MULTI_QUERY:
        base = MultiQueryRetriever.from_llm(retriever=base, llm=llm)
    return ContextualCompressionRetriever(base_compressor=build_reranker(), base_retriever=base)

retriever = build_retriever(docsearch, chunks, llm)
print("Retriever ready")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Retriever ready


## Conversational prompts with medical guardrails

In [13]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

system_prompt = """You are a careful medical information assistant for educational use.
Use ONLY the retrieved context below to answer the user's question.

Rules:
- If the answer is not in the context, say you don't know and recommend consulting a licensed healthcare professional. Never invent facts.
- Provide general information only. Do NOT diagnose, prescribe, or give personalised dosage instructions.
- Always remind the user to consult a qualified healthcare professional for personal medical concerns.
- Be concise (max ~5 sentences).

Context:
{context}"""

contextualize_q_system_prompt = """Given the chat history and the latest user question which might reference the chat history, formulate a standalone question that can be understood without the chat history. Do NOT answer it; just reformulate it if needed, otherwise return it unchanged."""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

## History-aware conversational RAG chain

In [14]:
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_prompt)
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)
print("RAG chain ready")

RAG chain ready


## Emergency gate, citations, grounding check

In [15]:
from langchain_core.messages import HumanMessage, AIMessage

EMERGENCY_KEYWORDS = [
    "chest pain", "can't breathe", "cannot breathe", "difficulty breathing",
    "suicidal", "suicide", "kill myself", "want to die", "self harm", "self-harm",
    "overdose", "unconscious", "stroke", "heart attack", "severe bleeding",
    "anaphylaxis", "seizure",
]
EMERGENCY_MESSAGE = (
    "This may be a medical emergency. I cannot help with urgent situations. "
    "Please call your local emergency number immediately (e.g. 911 in the US, 112 in the EU, 108 in India) "
    "or go to the nearest emergency department. If you are in crisis, contact a local crisis hotline now."
)
NO_CONTEXT_MESSAGE = (
    "I couldn't find information about that in my medical knowledge base, so I can't answer it reliably. "
    "Please consult a licensed healthcare professional."
)
GROUNDING_CAUTION = (
    "Note: parts of this answer may not be fully supported by my source documents. "
    "Please verify with a licensed healthcare professional."
)

def check_emergency(text):
    low = text.lower()
    return EMERGENCY_MESSAGE if any(k in low for k in EMERGENCY_KEYWORDS) else None

def format_sources(context):
    seen, sources = set(), []
    for d in context:
        meta = d.metadata or {}
        src = os.path.basename(str(meta.get("source", "unknown")))
        page = meta.get("page")
        key = (src, page)
        if key in seen:
            continue
        seen.add(key)
        sources.append({"source": src, "page": page, "snippet": d.page_content[:180].strip()})
    return sources

def verify_grounding(llm, answer, context):
    ctx = "\n\n".join(d.page_content for d in context)
    prompt = (
        "You are a strict fact-checker. Based ONLY on the CONTEXT, is the ANSWER fully supported? "
        "Reply with exactly 'yes' or 'no'.\n\nCONTEXT:\n" + ctx + "\n\nANSWER:\n" + answer
    )
    try:
        return llm.invoke(prompt).content.strip().lower().startswith("yes")
    except Exception:
        return True

def answer_question(question, chat_history=None):
    chat_history = chat_history or []
    emer = check_emergency(question)
    if emer:
        return {"answer": emer, "sources": [], "emergency": True}
    result = rag_chain.invoke({"input": question, "chat_history": chat_history})
    answer = result.get("answer", "")
    context = result.get("context", [])
    if not context:
        answer = NO_CONTEXT_MESSAGE
    elif GROUNDING_CHECK and not verify_grounding(llm, answer, context):
        answer = answer + "\n\n" + GROUNDING_CAUTION
    return {"answer": answer, "sources": format_sources(context), "emergency": False}

## Test it

In [16]:
res = answer_question("What is the treatment for acne?")
print(res["answer"])
print("\nSources:")
for s in res["sources"]:
    print(" -", s["source"], "(page", s["page"], ")")

The treatment for acne includes proper cleansing to keep the skin oil-free, eating a well-balanced diet, and avoiding certain foods and substances. Alternative treatments may also include supplementation with herbs and nutrients, as well as Chinese herbal remedies. Topical antifungal drugs are not typically used to treat acne. It's recommended to consult a qualified healthcare professional for personalized advice on treating acne, as they can provide guidance tailored to individual needs.

Sources:
 - Medical_book.pdf (page 283 )
 - Medical_book.pdf (page 39.0 )


In [17]:
# Feature 2: follow-up that depends on memory ("it" = acne)
history = [
    HumanMessage(content="What is acne?"),
    AIMessage(content="Acne is a common skin condition where pores become clogged."),
]
print(answer_question("What causes it?", history)["answer"])

The exact cause of acne is unknown, but several risk factors have been identified, including age, gender, hormonal disorders, heredity, hormonal changes, diet, certain drugs, personal hygiene, cosmetics, environment, and stress. For personalized information and advice, consult a qualified healthcare professional.


In [18]:
# Feature 9: emergency gate
print(answer_question("I have severe chest pain and can't breathe")["answer"])

This may be a medical emergency. I cannot help with urgent situations. Please call your local emergency number immediately (e.g. 911 in the US, 112 in the EU, 108 in India) or go to the nearest emergency department. If you are in crisis, contact a local crisis hotline now.


In [19]:
# Feature 10: out-of-domain question should not hallucinate
print(answer_question("Who won the 2022 FIFA World Cup?")["answer"])

I don't know who won the 2022 FIFA World Cup, as this information is not in the provided context. For the most accurate and up-to-date information, I recommend checking a reliable sports news source. If you have any medical-related questions, I'll be happy to help, and please consult a qualified healthcare professional for personal medical concerns.

Note: parts of this answer may not be fully supported by my source documents. Please verify with a licensed healthcare professional.
